In [ ]:
from sqlalchemy import create_engine
import pandas as pd

DATE = '2026-01-23'
PLATFORM = 1
INCIDENT_TYPES = ['later_than_following_tram', 'cancelled', 'major_delay']

SQL_QUERY = f"""
    SELECT datetime, realdatetime, delay FROM kruppallee.departures 
    WHERE
        DATE(datetime) = '{DATE}'
        AND line in ('107', '108')
        AND platform = {PLATFORM}
    ORDER BY datetime ASC
"""

DB_HOST = 'raspberrypi'  # from the SSH server's perspective
DB_PORT = 3306
DB_USER = 'python'
DB_PASSWORD = 'h6s73hb378f7wh'
DB_NAME = 'kruppallee'


# Build SQLAlchemy engine for MySQL
engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df = pd.read_sql(SQL_QUERY, con=engine)
df['time'] = pd.to_datetime(df['datetime']).dt.time
df['realtime'] = pd.to_datetime(df['realdatetime']).dt.time
df.drop(columns=['datetime', 'realdatetime'], inplace=True)
df.head()

,delay,time,realtime
0,0.0,04:40:00,04:40:00
1,-9999.0,05:00:00,NaT
2,0.0,05:20:00,05:20:00
3,0.0,05:40:00,05:40:00
4,0.0,05:56:00,05:56:00


In [2]:
df1 = df[df['realtime'].notna()]
df1 = df1.reset_index(drop=True)

df1['realindex'] = df1['realtime'].rank(method='first').astype(int) - 1
df2 = df1[df1.index != df1['realindex']]
df2

,delay,time,realtime,realindex
31,13.0,08:52:00,09:05:00,32
32,5.0,08:56:00,09:01:00,31
87,8.0,15:12:00,15:20:00,88
88,0.0,15:16:00,15:16:00,87
93,8.0,15:42:00,15:50:00,94
94,0.0,15:46:00,15:46:00,93
101,8.0,16:22:00,16:30:00,102
102,0.0,16:26:00,16:26:00,101
107,7.0,16:52:00,16:59:00,108
108,0.0,16:56:00,16:56:00,107


In [3]:
df3 = df[df['delay'] >= 10]
df3

,delay,time,realtime
8,11.0,06:36:00,06:47:00
24,13.0,08:02:00,08:15:00
25,15.0,08:06:00,08:21:00
27,10.0,08:16:00,08:26:00
28,18.0,08:22:00,08:40:00
29,16.0,08:26:00,08:42:00
30,17.0,08:32:00,08:49:00
31,14.0,08:36:00,08:50:00
33,15.0,08:46:00,09:01:00
34,13.0,08:52:00,09:05:00
